# Indoor Positioning with DICHASUS and TensorFlow
Train a TensorFlow-based deep neural network to perform indoor transmitter positioning in a well-known physical environment.

## The Dataset

In [ ]:
!mkdir dichasus
!wget --content-disposition https://darus.uni-stuttgart.de/api/access/datafile/:persistentId?persistentId=doi:10.18419/darus-2202/2 -P dichasus # dichasus-0152
!wget --content-disposition https://darus.uni-stuttgart.de/api/access/datafile/:persistentId?persistentId=doi:10.18419/darus-2202/3 -P dichasus # dichasus-0153
!wget --content-disposition https://darus.uni-stuttgart.de/api/access/datafile/:persistentId?persistentId=doi:10.18419/darus-2202/4 -P dichasus # dichasus-0154

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
import numpy as np
%matplotlib inline

training_files = ["dichasus/dichasus-0152.tfrecords", "dichasus/dichasus-0153.tfrecords"]
test_files = ["dichasus/dichasus-0154.tfrecords"]

def record_parse_function(proto):
	record = tf.io.parse_single_example(proto, {
		"csi": tf.io.FixedLenFeature([], tf.string, default_value = ''),
		"pos-tachy": tf.io.FixedLenFeature([], tf.string, default_value = '')
	})
	csi = tf.ensure_shape(tf.io.parse_tensor(record["csi"], out_type = tf.float32), (32, 1024, 2))
	pos_tachy = tf.ensure_shape(tf.io.parse_tensor(record["pos-tachy"], out_type = tf.float64), (3))

	# We only care about x/y position dimensions, z is always close to 0
	return csi, pos_tachy[:2]

training_set = tf.data.TFRecordDataset(training_files).map(record_parse_function)
test_set = tf.data.TFRecordDataset(test_files).map(record_parse_function)

### Positioning "Ground Truth" Information

In [ ]:
positions = []
for datapoint in test_set:
	positions.append(datapoint[1].numpy())
positions = np.vstack(positions)

plt.figure(figsize=(5, 6))
plt.scatter(positions[:, 0], positions[:, 1])
plt.axis("equal")
plt.xlabel("x coordinate [m]")
plt.ylabel("y coordinate [m]")
plt.show()

## Channel Coefficients

In [ ]:
csi_tensor = next(iter(test_set.shuffle(1000).take(1)))[0]

plt.figure(figsize = (10, 7))
plt.subplot(211)
plt.title("Channel State Information: Channel Coefficients")
plt.xlabel("Subcarrier $i$")
plt.ylabel("Referenced channel coeff.,\n abs. value $|h_i|_{dB}$ [dB]")
for antenna_csi in tf.unstack(csi_tensor):
	plt.plot(10 * np.log10(np.abs(np.fft.fftshift(antenna_csi[:,0]**2 + antenna_csi[:,1]**2))))

plt.subplot(212)
plt.xlabel("Subcarrier $i$")
plt.ylabel("Referenced channel coeff.,\nphase shift $arg(h_i)$")
for antenna_csi in tf.unstack(csi_tensor):
	plt.plot(np.unwrap(np.fft.fftshift(np.arctan2(antenna_csi[:,1], antenna_csi[:,0]))))

plt.show()

## Feature Engineering and Input Normalization

In [ ]:
def get_feature_mapping(chunksize = 32):
	def compute_features(csi, pos_tachy):
		assert(csi.shape[1] % chunksize == 0)
		featurecount = csi.shape[1] // chunksize
		csi_averaged = tf.stack([tf.math.reduce_mean(csi[:, (chunksize * s):(chunksize * (s + 1)), :], axis = 1) for s in range(featurecount)], axis = 1)
		powers = csi_averaged[:,:,0]**2 + csi_averaged[:,:,1]**2
		return csi_averaged, pos_tachy

	return compute_features

training_set_features = training_set.map(get_feature_mapping(32))
test_set_features = test_set.map(get_feature_mapping(32))

training_set_features = training_set_features.shuffle(buffer_size = 100000)
test_set_features = test_set_features.shuffle(buffer_size = 100000)

## Neural Network Architecture

In [ ]:
nn_input = tf.keras.Input(shape=(32,32,2), name="input")

nn_output = tf.keras.layers.Flatten()(nn_input)
nn_output = tf.keras.layers.Dense(units=512, activation="relu")(nn_output)
nn_output = tf.keras.layers.Dense(units=128, activation="relu")(nn_output)
nn_output = tf.keras.layers.Dense(units=128, activation="relu")(nn_output)
nn_output = tf.keras.layers.Dense(units=2, activation="linear", name="output")(nn_output)

nn = tf.keras.Model(inputs=nn_input, outputs=nn_output, name="localization")

## Training and Validation

In [ ]:
nn.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.MeanSquaredError())

batch_sizes = [32,64,256,1024,4096]
for b in batch_sizes:
	dataset_batched = training_set_features.batch(b)
	test_set_batched = test_set_features.batch(b)
	print("\nTraining with batch size:", b)
	nn.fit(dataset_batched, epochs=20, validation_data=test_set_batched)

## Evaluation

In [ ]:
true_positions = []
predictions = []

for channel_coeffs, positions in test_set_batched:
	true_positions.append(positions)
	predictions.append(nn.predict(channel_coeffs))

true_positions = np.vstack(true_positions)
predictions = np.vstack(predictions)

error_vectors = predictions - true_positions
absolute_errors = np.sqrt(error_vectors[:,0]**2 + error_vectors[:,1]**2)

In [ ]:
plt.figure(figsize=(12,3))
plt.title('Error distance distibution')
plt.xlabel('error distance in m')
plt.ylabel('occurences')

plt.hist(absolute_errors, bins=100, range=(0, 1))
plt.show()

In [ ]:
plt.figure(figsize=(12, 12))

plt.hexbin(true_positions[:,0], true_positions[:,1], C = absolute_errors, gridsize = 35, cmap = cm.jet, vmin = 0, vmax = 0.5)
plt.axis("equal")
cb = plt.colorbar()
cb.set_label("Absolute Error [m]", fontsize = 14)
plt.xlabel("x [m]", fontsize = 14)
plt.ylabel("y [m]", fontsize = 14)
plt.show()